# Aion config — loading, env overlays, and deep merge

This notebook uses **`aion.config`**: TOML/YAML files, `AION_*` environment variables, nested keys, layered files, and typed coercion for string values.

**Install extras:** `pip install aqwel-aion[config]` (PyYAML; on Python &lt; 3.11 also `tomli`).

Sample files in this folder: `sample.toml`, `sample_override.yaml`.

In [ ]:
from pathlib import Path
import aion.config as cfg

PKG = Path(cfg.__file__).resolve().parent
EXAMPLES = PKG / "examples"
SAMPLE_TOML = EXAMPLES / "sample.toml"
SAMPLE_YAML = EXAMPLES / "sample_override.yaml"
print("Examples dir:", EXAMPLES)

## 1. Load TOML and read nested keys with `get_nested`

In [ ]:
toml_cfg = cfg.load_toml_file(SAMPLE_TOML)
print("db.host:", cfg.get_nested(toml_cfg, "db.host"))
print("llm.timeout_seconds:", cfg.get_nested(toml_cfg, "llm.timeout_seconds"))
print("missing:", cfg.get_nested(toml_cfg, "nope.key", default="<default>"))

## 2. `set_nested` and `deep_merge`

In [ ]:
a = {"x": 1, "db": {"host": "a"}}
b = {"db": {"port": 99}, "y": 2}
merged = cfg.deep_merge(a, b)
merged

In [ ]:
mutable = {"app": {"name": "demo"}}
cfg.set_nested(mutable, "app.version", "0.1.9")
mutable

## 3. Layered configs (base TOML + override YAML)

In [ ]:
layered = cfg.load_layered(SAMPLE_TOML, SAMPLE_YAML, merge_env=False)
layered

## 4. Environment overrides (`AION_section__key`)

Temporarily set env vars, merge, then restore (optional exercise).

In [ ]:
import os

base = cfg.load_toml_file(SAMPLE_TOML)
os.environ["AION_DB__HOST"] = "from-env.example"
os.environ["AION_APP__DEBUG"] = "true"
cfg.merge_env_overrides(base, prefix="AION_")
print("db after env:", base["db"])
print("app.debug (string):", base.get("app", {}).get("debug"))

typed = cfg.coerce_string_values(base)
print("app.debug coerced:", typed["app"]["debug"], type(typed["app"]["debug"]))

del os.environ["AION_DB__HOST"]
del os.environ["AION_APP__DEBUG"]

## 5. `load_first_existing` and `require_keys`

In [ ]:
cfg_dict, path_used = cfg.load_first_existing(
    [EXAMPLES / "nonexistent.toml", SAMPLE_TOML],
    merge_env=False,
)
path_used, list(cfg_dict.keys())[:3]

In [ ]:
cfg.require_keys(layered, "db.host", "app.name")
print("require_keys passed")

## 6. Save YAML (`save_yaml_file`)

Writes a round-trippable snapshot (pick a temp path in your project).

In [ ]:
import tempfile

out = Path(tempfile.mkdtemp()) / "snapshot.yaml"
cfg.save_yaml_file(out, layered)
print(out.read_text()[:200], "...")